# Backline: one circuit, from laptop to FPGA and GPU

Fault-tolerant computation relies on a classical decoder that turns each round of syndrome
measurements into corrections. Syndromes arrive every QEC cycle, about 1 µs on superconducting and
photonic hardware, and the decoder has to keep pace: some operations cannot proceed until it has
caught up, so a decoder that falls behind slows the whole computation down exponentially.

To keep pace, a primary decoder runs on an FPGA or ASIC right next to the quantum hardware and
handles most errors in under a microsecond. Rarer, harder error patterns need more compute than it
has, so they are handed to a backup decoder on a CPU or GPU. That hand-off is a round trip between
two machines, and it has to take microseconds, not milliseconds, or the quantum hardware stalls.

Backline lets us write the program once in PennyLane and choose where each piece runs and how they
talk. We take one GHZ circuit through three setups, changing only the hardware description each
time:

1. a plain PennyLane Lightning simulator,
2. a Steane-encoded run with the decoder on this laptop's CPU,
3. the same encoded run with an FPGA controller and a GPU decoder talking over RDMA.


In [ ]:
import pennylane as qp

## 1. Simulate it

A three-qubit GHZ state, compiled with Catalyst (`qjit`) and run on the Lightning simulator.
We expect every shot to be `000` or `111`, with roughly equal counts.

In [ ]:
dev = qp.device("lightning.qubit", wires=3)

@qp.qjit(capture=True)
@qp.set_shots(10)
@qp.qnode(dev, mcm_method="one-shot")
def ghz():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    qp.CNOT([1, 2])
    return qp.sample([qp.measure(0), qp.measure(1), qp.measure(2)])


print("samples:", ghz())

samples: [[1 1 1]
 [1 1 1]
 [1 1 1]
 [1 1 1]
 [0 0 0]
 [1 1 1]
 [0 0 0]
 [1 1 1]
 [0 0 0]
 [0 0 0]]


## 2. Add error correction, still on this laptop

Now we replace the device with a `Backline`, which has two roles:

- **controller**: runs the quantum program, here on the same Lightning simulator;
- **coprocessor**: runs the decoder, here a precompiled Steane decoder on this CPU.

`qec_code="steane"` asks the compiler to encode each logical qubit in seven physical qubits and
to add syndrome extraction, decoding and correction around every logical gate. We don't write any
of that by hand. Each decode is a round trip from the controller to the coprocessor.

With `transport="memcpy"` both roles live in this process, so no network or special hardware is
needed. The circuit itself is unchanged.

In [ ]:
from placement import STEANE_CPU_DECODER_LIB_PATH

# The decoder: a C function in a shared library that Catalyst builds.
steane_decode = qp.CoprocessorFunction("steane_coprocessor", lib_path=STEANE_CPU_DECODER_LIB_PATH)

# The controller runs the circuit; the coprocessor answers each syndrome with a correction.
ctrl = qp.Controller(name="cpu-controller", device=qp.device("lightning.qubit", wires=3))
coproc = qp.Coprocessor(name="cpu-coproc", coprocessor_fn=steane_decode)

# memcpy: both roles in this process. qec_code: the compiler inserts encoding and decode rounds.
dev = qp.Backline(controller=ctrl, coprocessors=[coproc], transport="memcpy", qec_code="steane")

@qp.qjit(capture=True)
@qp.set_shots(10)
@qp.qnode(dev, mcm_method="one-shot")
def ghz():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    qp.CNOT([1, 2])
    return qp.sample([qp.measure(0), qp.measure(1), qp.measure(2)])


print("samples:", ghz())

samples: [[0 0 0]
 [1 1 1]
 [1 1 1]
 [0 0 0]
 [1 1 1]
 [1 1 1]
 [1 1 1]
 [0 0 0]
 [0 0 0]
 [0 0 0]]


## 3. Same program, real hardware

Finally, we move to the arrangement from the introduction, with the FPGA next to the quantum
hardware and a GPU decoder reached over the network. Here the GPU decodes every round, standing in
for the backup decoder:

- the **controller** goes to an AMD Versal VPK120 FPGA, whose fabric sends each syndrome and
  detects the reply in hardware, with no host CPU involved;
- the **decoder** becomes a GPU kernel on a remote server's AMD Instinct MI210, launched once
  and left running so that no round pays a kernel launch;
- the two talk over **RDMA**, directly across the network.

Only the node definitions change. Hosts, ports and devices come from a pre-configured
`machines.toml`.

The FPGA has no quantum hardware attached, so we give the controller no device and it falls back
to `null.qubit`, a mock device that accepts every gate without simulating it and returns 0 for
every measurement. So every sample is `000`.

That is deliberate. With no simulation running on the board, the run measures only the classical
path: syndrome out over RDMA, decode on the GPU, correction back. The board timestamps each of these
round trips in hardware (16,000 of them for 1,000 shots), and the table it prints is the result we
care about, around 4.4 µs at the median.

In [ ]:
from placement import REMOTE_GPU_COPROC, REMOTE_VPK_CTRL

# Controller on the FPGA board, with no device given, so it runs null.qubit.
# Decoder as a precompiled HIP kernel on the remote GPU.
ctrl = qp.Controller(name="fpga-controller", **REMOTE_VPK_CTRL)
coproc = qp.Coprocessor(name="gpu-coproc", coprocessor_fn="gpu_steane_launcher", **REMOTE_GPU_COPROC)

# Same Backline as before, with the transport switched to RDMA.
dev = qp.Backline(controller=ctrl, coprocessors=[coproc], transport="rdma", qec_code="steane")

@qp.qjit(capture=True)
@qp.set_shots(1000)
@qp.qnode(dev, mcm_method="one-shot")
def ghz():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    qp.CNOT([1, 2])
    return qp.sample([qp.measure(0), qp.measure(1), qp.measure(2)])


print("samples:", ghz())

[fpga-controller] catalyst-executor: loaded /home/petalinux/catalyst-exec/librt_transport.so
[fpga-controller] catalyst-executor: loaded /home/petalinux/catalyst-exec/librt_capi.so
[fpga-controller] catalyst-executor: loaded /home/petalinux/catalyst-exec/librtd_null_qubit.so
[fpga-controller] Listening on 127.0.0.1:7811
[fpga-controller] [127.0.0.1:7811] executor ready, waiting for connections
scp: Connection closed
[gpu-coproc] catalyst-executor: loaded /home/catalyst-exec/librt_transport.so
[gpu-coproc] catalyst-executor: loaded /home/catalyst-exec/librt_capi.so
[gpu-coproc] Listening on 127.0.0.1:7813
[gpu-coproc] [127.0.0.1:7813#3133915] Accepted connection
[fpga-controller] [127.0.0.1:7811] accepted connection from 127.0.0.1:39924 on pid 1209, waiting for next connection
[fpga-controller]
[fpga-controller]=== engine RTT (n=16000, 0 warmup dropped, hardware handshake) ===
[fpga-controller]   min          4245 ns
[fpga-controller]   p50          4415 ns
[fpga-controller]   p95      